# Lab 21 · The finale: check the brief, then hear it work

**Time:** 25 min · **Work in teams of 3**

**Where we are in the day:** Lab 20 gave us a confidence rule that asks back when the model is unsure; all five rows of the scoreboard are now saved.
**Today's question:** does the finished system meet the brief we wrote in Lab 16, and does it work on spoken commands?
**What you will do:**
- Put every number of the day against the four targets of the brief.
- Connect the pieces into one assistant: understand, decide, check, act.
- Speak to it: a speech model (Whisper) listens to short recordings, and you see where the time goes.

**What you will see:** in our test run three of the four targets were met and accuracy was just short (89.5 per cent against 90), and the spoken clips took between about 0.2 and 0.6 seconds each from speech to understanding.
**Why it matters on a device:** a product is judged on the whole chain and against its brief, not on the best single number.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **25 seconds** in all, once the speech model is on the disk.
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer. The first run downloads the speech model, which can take a few minutes.
- Before you start: Labs 16 to 20 must have been run, because this lab reads the five rows of the scoreboard and loads the merged model. If a row is missing, the first cell tells you.

**Words used in this lab:**
- **Speech recognition:** turning a recording into text. **Whisper** is the speech model we use.
- **Pipeline:** the chain of stages, here speech, then understanding, then action.
- **Waterfall:** a table of how many milliseconds each stage took.

**What you built today, step by step:**

| Step | What changed | Lab |
|---|---|---|
| 1 | the prompt | 16 |
| 2 | the model's skill (fine-tuned) | 17 |
| 3 | the model's size (quantised, runs on a CPU) | 18 |
| 4 | how it is served (many people at once) | 19 |
| 5 | what happens when it is wrong | 20 |

**Python in this lab:** loops, `if` and a few functions. The speech helper is in `edge_voice.py`.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**.*

In [ ]:
import sys, os, time
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import torch
import pandas as pd
import edge_lab as el
import edge_nlu as nlu
import edge_confidence as ec
import edge_voice as ev
import home_domain as hd
from slm_lab import check, OUT

In [ ]:
MERGED_FOLDER = str(OUT / "edge_merged")
board = el.load_scoreboard()
for step in board:
    print(step, board[step])

In [ ]:
check(len(board) >= 5, "Checkpoint: the scoreboard has all five rows (Labs 16 to 20).",
      "A row is missing. Look at the names printed above, run the lab of the missing row (16 to 20), then run this notebook again.")

**You should see** the rows saved by Labs 16 to 20. If a row is missing, run that lab first.

## 21.1 The scoreboard against the brief
In Lab 16 we wrote four targets. Here is what we reached. **Predict.** Write on paper which of the four targets you think were met (accuracy, memory, answer time, people at once).

In [ ]:
BRIEF = {"accuracy_percent": 90, "answer_ms": 500, "memory_gb": 2.0, "users_at_once": 32}    # the targets from Lab 16
checks = []
accuracy = board["4 served"]["exact_percent"]
checks.append(["accuracy", str(BRIEF["accuracy_percent"]) + " %", str(accuracy) + " % (all 72 messages)", accuracy >= BRIEF["accuracy_percent"]])
memory = board["3 quantised"]["file_mb"] / 1000
checks.append(["memory", str(BRIEF["memory_gb"]) + " GB", str(round(memory, 2)) + " GB (file size)", memory <= BRIEF["memory_gb"]])
answer = board["4 served"]["p95_ms"]
checks.append(["answer time", str(BRIEF["answer_ms"]) + " ms", str(answer) + " ms (slow wait, p95)", answer <= BRIEF["answer_ms"]])
people = board["4 served"]["people_at_once"]
checks.append(["people at once", str(BRIEF["users_at_once"]), str(people), people >= BRIEF["users_at_once"]])

Now print each target next to the result.

In [ ]:
for name, target, result, ok in checks:
    if ok:
        mark = "met"
    else:
        mark = "NOT met"
    print(name.ljust(16), target.ljust(10), result.ljust(40), mark)

In [ ]:
check(len(checks) == 4, "Checkpoint: all four targets of the brief were compared.",
      "Run the cell above again. If it stops with a KeyError, a scoreboard row is missing: run the lab that makes it.")

The accuracy comes from the full test of 72 messages in Lab 19 (the 16-bit model). The quantised model was tested on a sample of only 24 messages in Lab 18: that number is too small to put on a scoreboard. **You should see** which targets are met and which are not. In our test run: accuracy 89.5 per cent against 90 (not met, by half a point), memory 0.99 GB against 2.0 (met), answer time 383 ms against 500 (met) and 128 people at once against 32 (met). **An honest scoreboard is the product of an engineer.** A target that is not met is not a failure of the day: it tells you what the next iteration is (more training data, a different format, a bigger device).

## 21.2 One assistant, all the pieces
The pieces we have: a model that reads messages (Lab 17), a confidence rule (Lab 20), and the home from Day 2 with its safe functions. The assistant uses them in order. *In one sentence: the model proposes, and the code around it decides.*

1. **Understand:** the model reads the message and tells us how sure it is.
2. **Decide:** if it is not sure, **ask back**. If it is sure, continue.
3. **Check:** the home's own code validates the details (this is Day 2: the model proposes, the code decides).
4. **Act:** change the home.

In [ ]:
tok, model = el.load_edge_model(MERGED_FOLDER)
CONFIDENCE = 0.9                                 # 🔧 the threshold of Lab 20
home = hd.new_home()

Step 1, Understand: this function asks the model, and measures how sure it is and how long it took.

In [ ]:
def understand(text):
    started = time.time()
    probabilities = ec.label_probabilities(tok, model, nlu.SHORT_SYSTEM, text)
    best, best_p, second, second_p = ec.best_two(probabilities)
    prompt_text = el.make_prompt_text(tok, nlu.SHORT_SYSTEM, text)
    reply = el.ask_many(tok, model, [prompt_text])["replies"][0]
    understood_ms = round(1000 * (time.time() - started))
    return best, best_p, second, reply, understood_ms

Steps 2 to 4: decide, check and act.

In [ ]:
def assistant_step(text, home):
    best, best_p, second, reply, understood_ms = understand(text)
    if best_p < CONFIDENCE:
        return home, "Sorry, did you mean " + best + " or " + second + "?", understood_ms, 0
    slots = nlu.read_slots(reply)
    if best not in ["set_light", "set_ac", "set_blinds", "run_scene"]:
        return home, "(not a home action: " + best + ")", understood_ms, 0
    acted = time.time()
    clean, error = hd.validate_call(best, slots)
    if error is not None:
        return home, "I need more details: " + error, understood_ms, 0
    new_home, message = hd.execute(home, best, clean)
    return new_home, message, understood_ms, round(1000 * (time.time() - acted))

**You should see** nothing printed: we only defined the functions. Now we run them on messages, including noisy and Hinglish ones.

In [ ]:
messages = ["turn on the kitchen light", "tern on the livin room lite", "bedroom ki light jalao", "set the bedroom ac to 23",
            "close the study blinds", "start movie night", "what does CH38 mean", "play some music"]
for message in messages:
    home, answer, understood_ms, act_ms = assistant_step(message, home)
    print(message.ljust(30), "->", answer, "(" + str(understood_ms) + " ms)")

In [ ]:
check(len(home) > 0, "Checkpoint: the assistant ran on 8 messages and the home is still a valid state.",
      "Run the cell that defines the two functions (understand and assistant_step), then this cell again.")

**You should see** the home changing for real messages (lights, air conditioner, blinds, scene), a question or a note for the rest, and the time the understanding took (it varies with your GPU). Read the answers: where did the assistant ask back, and was that sensible?

## 21.3 Now speak to it
A voice assistant has a **speech recognition** stage before understanding. We use **Whisper** (a small speech model) on short recordings. The text it hears may contain mistakes, exactly like the noisy messages of the test set. *In one sentence: the assistant now hears the user, so it also inherits the listening mistakes.*

If the folder `data/voice` has recorded `.wav` files, we use them. Otherwise the helper makes a few spoken clips with a text-to-speech model and adds a little noise.

In [ ]:
paths, source = ev.find_clips()
print(len(paths), "clips,", source)
listener = ev.load_listener()
print("speech recognition model loaded")

**You should see** the number of clips and the word `recorded` or `generated`. Now the whole pipeline: **speech → text → understanding → action**, with the time of each stage.

In [ ]:
voice_rows = []
for path in paths:
    heard, listen_seconds = ev.transcribe(listener, path)
    home, answer, understood_ms, act_ms = assistant_step(heard, home)
    voice_rows.append({"clip": os.path.basename(path), "heard": heard, "answer": answer, "speech ms": round(1000 * listen_seconds),
                       "understand ms": understood_ms, "act ms": act_ms})
pd.DataFrame(voice_rows)

In [ ]:
check(len(voice_rows) == len(paths) and len(paths) > 0, "Checkpoint: every clip went through the whole pipeline.",
      "No clips were processed. Run the cell that loads the clips and the listener (21.3) first, then this cell.")

**You should see** for each clip what Whisper heard, what the assistant did, and a timing **waterfall**: speech recognition, understanding, action. **Where does most of the time go?** Write down your answer from the table.

In our test run (six clips spoken by the trainer, Whisper small on the GPU): speech recognition took 110 to 560 ms per clip (the first clip is slower because the model is warming up), understanding 250 to 560 ms, and the action almost nothing. Five clips were heard exactly. One was heard as "Set the bedroom RC to 23" instead of "AC": a real speech mistake. Look at what the assistant did with it. Your clips may differ.

So the stages had similar sizes, and **neither one alone dominated.** If it is not the part we optimised today, that is the next thing to work on. A real product measures the **whole** chain, not only its own part.

## 21.4 The home at the end

In [ ]:
for name in home:
    if home[name] != "off" and home[name] != "closed":
        print(name, "=", home[name])

## 21.5 What you built today

In [ ]:
rows = []
for step in board:
    rows.append({"step": step, "what": board[step].get("what", "")})
pd.DataFrame(rows)

In [ ]:
check(len(rows) >= 5, "Checkpoint: five steps are on your final scoreboard. You have finished the day.",
      "Fewer than five rows. Run the labs that are missing (16 to 20), then run this notebook again.")

## What you take away
- Write the **brief** as numbers first, change the cheapest thing first (prompt, then skill, then size, then serving), and measure every step on a test set the model never saw.
- A model is never always right: **measure its confidence, ask back, fall back, and keep the mistakes**.
- The code around the model (validation, limits, approval) is what makes it safe, and a product is measured on the whole chain, not on one stage.

## Check your understanding
1. Which target of the brief was hardest to meet, and what would you try next?
*✍️ Your answer:* 
2. In the voice pipeline, which stage takes the most time, and does that match what we optimised today?
*✍️ Your answer:* 
3. The assistant asked back for a message. Was that a mistake of the system?
*✍️ Your answer:* 

## The end
There is no next lab. The bridge is your own project: pick one appliance or one workflow, write its brief as numbers, and repeat the five steps.

You started the day with a model that could not read a messy voice command. You now have a small, fast, measured and careful voice assistant on the edge. **You can do this for your own appliances.**